# Pemeringkatan Influencer: Entropy Weight + EDAS

Disusun per **BLOK** (samaan kayak notebook scoring sebelumnya), tiap blok ada cell bernomor.

| Blok | Isi |
|---|---|
| **A** | Setup, config, load semua sumber data (skor sentimen, followers, metadata video) |
| **B** | Hitung C2 (followers), C4 (rata-rata shares), C5 (engagement rate) per influencer |
| **C** | Susun matriks keputusan (gabung C1+C2+C4+C5) + filter followers & threshold komentar |
| **D** | Fungsi Entropy Weight (Persamaan 2.2-2.6 proposal) |
| **E** | Fungsi EDAS (Persamaan 2.7-2.14 proposal) |
| **F** | Jalanin semua kombinasi skenario (3 skema sentimen x 3 threshold komentar) + rangking |

**Catatan skenario yang disepakati:**
- Kriteria: **C1** (sentimen), **C2** (followers), **C4** (rata-rata shares), **C5** (engagement rate = rata-rata total interaksi / followers). **C3 (views) di-drop** karena data views ga kesimpen pas scraping -- ini perlu disebut sebagai keterbatasan metodologi di skripsi.
- C1 dicoba 3 varian: `3class` (positif=+1/netral=0/negatif=-1), `3class_hanya_positif` (positif=+1/netral=0/negatif=0), `binary` (pake varian B: negatif=-1/non-negatif=+1).
- Threshold minimal komentar bersih per influencer dicoba 3 skenario: **30, 50, 100**.
- Filter tambahan dari batasan masalah proposal: **followers >= 10.000**.
- Filter tambahan lagi: **minimal 10 video** per influencer -- biar C4 (rata-rata shares) & C5 (engagement rate) ga dihitung dari sample video yang kelewat sedikit (rawan di-drag 1 video viral jadi outlier, seperti kasus `farhanzubedi` yang ketahuan di Cell F5/F6).
- Total: **3 x 3 = 9 kombinasi EDAS** yang dijalanin & dibandingin.

---
## BLOK A — Setup, Config, Load Data

**Cell A1**: import & konfigurasi (path, daftar skema C1, daftar threshold, min followers).
**Cell A2**: cari folder root repo + susun semua path.
**Cell A3**: load `skor_influencer.csv`, `metadata_profil.csv`, `metadata_video.csv`.

In [17]:
# ==== Cell A1: Import & Konfigurasi ====
import numpy as np
import pandas as pd
from pathlib import Path

EXCLUDE_USERNAMES = ['drrichardlee']   # konsisten sama tahap scoring sebelumnya

# --- 3 skema C1 yang mau dicoba ---
SKEMA_C1 = {
    '3class': 'C1_3class',
    '3class_hanya_positif': 'C1_3class_hanya_positif',
    'binary': 'C1_binary_varianB',
}

# --- 3 threshold minimal komentar bersih per influencer yang mau dicoba ---
DAFTAR_THRESHOLD_KOMENTAR = [30, 50, 100]

# --- Batasan dari proposal (bab 1.3): influencer minimal 10.000 followers ---
MIN_FOLLOWERS = 10_000

# --- Filter tambahan: minimal 10 video, biar C4/C5 (dihitung dari rata-rata per video) ---
# --- ga rawan di-drag outlier dari 1-2 video viral doang ---
MIN_VIDEO = 10

print('Config siap.')
print('Skema C1:', list(SKEMA_C1.keys()))
print('Threshold komentar yg dicoba:', DAFTAR_THRESHOLD_KOMENTAR)
print('Min followers:', MIN_FOLLOWERS)
print('Min jumlah video:', MIN_VIDEO)


Config siap.
Skema C1: ['3class', '3class_hanya_positif', 'binary']
Threshold komentar yg dicoba: [30, 50, 100]
Min followers: 10000
Min jumlah video: 10


In [18]:
# ==== Cell A2: Cari folder root repo + path ====
def find_repo_root():
    current = Path.cwd()
    for _ in range(6):
        if (current / '01_pengumpulan_data').is_dir() and (current / '03_Indobert').is_dir():
            return current
        if current.parent == current:
            break
        current = current.parent
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Nggak ketemu folder root repo (butuh '01_pengumpulan_data' dan '03_Indobert' "
        "keduanya ada sebagai subfolder). Jalankan dari dalam folder repo."
    )
print('Repo root:', REPO_ROOT)

PATH_SKOR_INFLUENCER = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_scoring' / 'skor_influencer.csv'
PATH_PROFIL = REPO_ROOT / '01_pengumpulan_data' / '04_1_mendapatkan_metadata_akun' / 'metadata_profil.csv'
PATH_VIDEO = REPO_ROOT / '01_pengumpulan_data' / '04_1_mendapatkan_metadata_akun' / 'metadata_video.csv'
OUT_DIR = REPO_ROOT / '03_Indobert' / 'Data' / 'hasil_edas'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Skor influencer :', PATH_SKOR_INFLUENCER)
print('Profil (followers):', PATH_PROFIL)
print('Video (shares dkk):', PATH_VIDEO)
print('Output dir      :', OUT_DIR)


Repo root: c:\Users\nesha\Tugas_akhir\seleksi-influencer
Skor influencer : c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_scoring\skor_influencer.csv
Profil (followers): c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\04_1_mendapatkan_metadata_akun\metadata_profil.csv
Video (shares dkk): c:\Users\nesha\Tugas_akhir\seleksi-influencer\01_pengumpulan_data\04_1_mendapatkan_metadata_akun\metadata_video.csv
Output dir      : c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_edas


In [19]:
# ==== Cell A3: Load 3 sumber data ====
skor_influencer = pd.read_csv(PATH_SKOR_INFLUENCER, dtype=str, keep_default_na=False, encoding='utf-8')
# kolom numerik di skor_influencer perlu di-convert balik (tadinya disimpen sbg string biar aman)
for kol in skor_influencer.columns:
    if kol != 'username':
        skor_influencer[kol] = pd.to_numeric(skor_influencer[kol], errors='coerce')

df_profil = pd.read_csv(PATH_PROFIL, dtype=str, keep_default_na=False, encoding='utf-8')
df_video = pd.read_csv(PATH_VIDEO, dtype=str, keep_default_na=False, encoding='utf-8')

for nama, df, req in [
    ('skor_influencer.csv', skor_influencer, {'username'} | set(SKEMA_C1.values())),
    ('metadata_profil.csv', df_profil, {'username', 'followers', 'scraped_at'}),
    ('metadata_video.csv', df_video, {'username', 'video_id', 'like_count', 'comment_count', 'share_count', 'save_count'}),
]:
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f'{nama} kurang kolom: {missing}')

print(f'skor_influencer : {len(skor_influencer):,} influencer')
print(f'metadata_profil : {len(df_profil):,} baris')
print(f'metadata_video  : {len(df_video):,} baris')


skor_influencer : 42 influencer
metadata_profil : 47 baris
metadata_video  : 6,939 baris


---
## BLOK B — Hitung C2 (followers), C4 (rata-rata shares), C5 (engagement rate)

**Cell B1**: ambil followers per influencer (dedup kalau ada baris duplikat, ambil yang paling baru).
**Cell B2**: agregasi `metadata_video.csv` -> rata-rata shares (C4) & rata-rata total interaksi per influencer (buat numerator C5).
**Cell B3**: hitung C5 = rata-rata total interaksi / followers.

In [20]:
# ==== Cell B1: Ambil followers per influencer ====
def ambil_followers(df_profil, exclude_usernames):
    df = df_profil[~df_profil['username'].isin(exclude_usernames)].copy()
    df['followers'] = pd.to_numeric(df['followers'], errors='coerce')
    # kalau 1 username muncul >1 baris (di-scrape ulang beberapa kali), ambil yg paling baru
    df = df.sort_values('scraped_at').drop_duplicates(subset='username', keep='last')
    return df[['username', 'followers']].reset_index(drop=True)


followers_df = ambil_followers(df_profil, EXCLUDE_USERNAMES)
print(f'Followers ter-load: {len(followers_df)} influencer')
followers_df.sort_values('followers', ascending=False).head(10)


Followers ter-load: 46 influencer


,username,followers
34,\drrichardlee,6400000.0
39,drkevinmak,4200000.0
1,tirtacipeng,2800000.0
27,dokter_ika,2600000.0
40,farhanzubedi,2500000.0
28,dokteramiraobgyn,2300000.0
24,aymanalts,2000000.0
32,dr.ziee,1800000.0
14,ekidarehanf,1400000.0
6,nadialaydrus,1400000.0


In [21]:
# ==== Cell B2: Agregasi metadata_video.csv -> C4 & numerator C5 ====
def agregasi_video(df_video, exclude_usernames):
    df = df_video[~df_video['username'].isin(exclude_usernames)].copy()
    for kol in ['like_count', 'comment_count', 'share_count', 'save_count']:
        df[kol] = pd.to_numeric(df[kol], errors='coerce').fillna(0)
    df['total_interaksi'] = df['like_count'] + df['comment_count'] + df['share_count'] + df['save_count']
    agg = df.groupby('username').agg(
        C4_rata_shares=('share_count', 'mean'),
        rata_total_interaksi=('total_interaksi', 'mean'),
        jumlah_video=('video_id', 'nunique'),
    ).reset_index()
    return agg


video_agg = agregasi_video(df_video, EXCLUDE_USERNAMES)
print(f'Video agg ter-load: {len(video_agg)} influencer')
video_agg.sort_values('C4_rata_shares', ascending=False).head(10)


Video agg ter-load: 45 influencer


,username,C4_rata_shares,rata_total_interaksi,jumlah_video
26,farhanzubedi,56821.200000,369621.400000,5
42,tirtacipeng,8831.398406,81135.764940,251
31,ikhsanqthii,4914.705882,75865.875294,425
33,jamilatussadiyah,3516.000000,12688.384615,39
29,giapratamamd,3467.525180,66744.208633,139
41,spesialist.kesehatan,2984.300000,5647.700000,10
32,itsdoktermuda,2542.267380,20667.294118,187
23,drkevinmak,2525.886266,19575.105150,466
25,ekidarehanf,1751.465241,14941.689840,187
11,doktermedok,1391.747253,8931.148352,182


In [22]:
# ==== Cell B3: Gabungin followers + video_agg, hitung C5 (engagement rate) ====
metrik_akun = followers_df.merge(video_agg, on='username', how='inner')
metrik_akun['C5_engagement_rate'] = metrik_akun['rata_total_interaksi'] / metrik_akun['followers']

print(f'Metrik akun lengkap: {len(metrik_akun)} influencer '
      f'(irisan yg ada DI followers.csv MAUPUN video.csv)')
metrik_akun.sort_values('C5_engagement_rate', ascending=False).head(10)


Metrik akun lengkap: 45 influencer (irisan yg ada DI followers.csv MAUPUN video.csv)


,username,followers,C4_rata_shares,rata_total_interaksi,jumlah_video,C5_engagement_rate
17,fauzaannibras,13200.0,545.480447,5926.357542,179,0.448966
0,spesialist.kesehatan,16000.0,2984.300000,5647.700000,10,0.352981
39,farhanzubedi,2500000.0,56821.200000,369621.400000,5,0.147849
34,itsdoktermuda,217300.0,2542.267380,20667.294118,187,0.095109
42,ikhsanqthii,974300.0,4914.705882,75865.875294,425,0.077867
43,jamilatussadiyah,202500.0,3516.000000,12688.384615,39,0.062659
40,gelarsramdhani,12800.0,50.495495,718.270270,111,0.056115
41,giapratamamd,1300000.0,3467.525180,66744.208633,139,0.051342
35,dokter.harmonospanak,19500.0,148.443925,611.065421,214,0.031337
1,tirtacipeng,2800000.0,8831.398406,81135.764940,251,0.028977


---
## BLOK C — Susun Matriks Keputusan + Filter

**Cell C1**: fungsi `susun_matriks()` -- gabungin C1 (sesuai skema yg dipilih) + C2 + C4 + C5, filter followers >= 10.000, jumlah video >= 10, & jumlah komentar >= threshold.
**Cell C2**: contoh coba 1 kombinasi (skema `3class`, threshold 30) buat ngecek matriksnya kebentuk bener.

In [23]:
# ==== Cell C1: Fungsi susun matriks keputusan ====
def susun_matriks(skor_influencer_df, nama_skema, metrik_akun_df,
                   min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO):
    """Gabungin C1 (sesuai skema) + C2/C4/C5 dari metrik_akun, lalu filter followers, jumlah
    video, & jumlah komentar. Return: df matriks (index=username, kolom C1/C2/C4/C5) + jumlah
    yg kefilter.

    Filter jumlah video penting karena C4/C5 dihitung sbg RATA-RATA per video -- kalau cuma
    dihitung dari sedikit video, rata-ratanya rawan di-drag jauh sama 1 video yg kebetulan
    viral (lihat kasus farhanzubedi di Cell F5/F6)."""
    kolom_c1 = SKEMA_C1[nama_skema]
    kolom_p = 'P_' + kolom_c1.replace('C1_', '')

    df = skor_influencer_df[['username', kolom_c1, kolom_p]].merge(
        metrik_akun_df, on='username', how='inner')

    sebelum = len(df)
    df = df[df['followers'] >= min_followers]
    df = df[df['jumlah_video'] >= min_video]
    df = df[df[kolom_p] >= min_komentar]
    sesudah = len(df)

    matriks = df.rename(columns={
        kolom_c1: 'C1', 'followers': 'C2', 'C4_rata_shares': 'C4', 'C5_engagement_rate': 'C5',
    }).set_index('username')[['C1', 'C2', 'C4', 'C5']]

    return matriks, sebelum, sesudah


print('Fungsi susun_matriks() siap.')


Fungsi susun_matriks() siap.


In [24]:
# ==== Cell C2: Coba 1 kombinasi dulu (skema 3class, threshold 30) ====
matriks_contoh, sebelum, sesudah = susun_matriks(
    skor_influencer, '3class', metrik_akun,
    min_followers=MIN_FOLLOWERS, min_komentar=30, min_video=MIN_VIDEO)
print(f'Influencer sebelum filter: {sebelum}, sesudah filter (followers>={MIN_FOLLOWERS}, '
      f'video>={MIN_VIDEO}, komentar>=30): {sesudah}')
matriks_contoh.sort_values('C1', ascending=False)


Influencer sebelum filter: 42, sesudah filter (followers>=10000, video>=10, komentar>=30): 37


,C1,C2,C4,C5
username,,,,
dokter.harmonospanak,0.326241,19500.0,148.443925,0.031337
reisabrotoasmoro,0.317757,207600.0,69.734513,0.003666
doktermedok,0.184089,841200.0,1391.747253,0.010617
doktermatahendra,0.178161,154100.0,73.957983,0.002818
giapratamamd,0.168188,1300000.0,3467.525180,0.051342
dr.ziee,0.148464,1800000.0,321.154639,0.003889
gelarsramdhani,0.146667,12800.0,50.495495,0.056115
meta.hanindita,0.112903,68000.0,4.800000,0.001707
doktervito,0.107143,53000.0,13.378788,0.002228


---
## BLOK D — Entropy Weight (Persamaan 2.2 - 2.6 proposal)

**Cell D1**: fungsi `hitung_bobot_entropy()` -- normalisasi probabilitas, hitung entropy $E_j$, derajat diversifikasi $d_j$, bobot $w_j$.

In [25]:
# ==== Cell D1: Fungsi Entropy Weight ====
def hitung_bobot_entropy(matriks, kolom_kriteria):
    """Persamaan 2.2 (normalisasi pij), 2.3 (entropy Ej), 2.4 (konstanta k), 2.5 (diversifikasi
    dj), 2.6 (bobot wj). Kalau pij=0, pij*ln(pij) dianggap 0 (sesuai catatan di proposal)."""
    m = len(matriks)
    k = 1 / np.log(m)

    detail = []
    for kol in kolom_kriteria:
        x = matriks[kol].values.astype(float)
        p = x / x.sum()
        p_lnp = np.where(p > 0, p * np.log(p), 0.0)
        Ej = -k * p_lnp.sum()
        dj = 1 - Ej
        detail.append({'kriteria': kol, 'Ej': Ej, 'dj': dj})

    total_d = sum(d['dj'] for d in detail)
    bobot = {}
    for d in detail:
        d['wj'] = d['dj'] / total_d
        bobot[d['kriteria']] = d['wj']

    return bobot, pd.DataFrame(detail).set_index('kriteria')


print('Fungsi hitung_bobot_entropy() siap.')

# Coba langsung ke matriks contoh dari Cell C2
bobot_contoh, detail_entropy_contoh = hitung_bobot_entropy(matriks_contoh, ['C1', 'C2', 'C4', 'C5'])
print('\nBobot entropy (contoh, skema 3class, threshold 30):')
print(detail_entropy_contoh.to_string())
print('\nTotal bobot (harus = 1.0):', sum(bobot_contoh.values()))


Fungsi hitung_bobot_entropy() siap.

Bobot entropy (contoh, skema 3class, threshold 30):
                Ej        dj        wj
kriteria                              
C1        0.910943  0.089057  0.105879
C2        0.819240  0.180760  0.214904
C4        0.755407  0.244593  0.290794
C5        0.673290  0.326710  0.388422

Total bobot (harus = 1.0): 1.0


C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: invalid value encountered in log
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)


---
## BLOK E — EDAS (Persamaan 2.7 - 2.14 proposal)

**Cell E1**: fungsi `hitung_edas()` -- rata-rata kriteria (AV), jarak positif/negatif dari rata-rata (PDA/NDA), skor berbobot (SP/SN), normalisasi (NSP/NSN), skor akhir (Appraisement Score / AS) + rangking.

In [26]:
# ==== Cell E1: Fungsi EDAS ====
def hitung_edas(matriks, kolom_kriteria, bobot):
    """Semua kriteria di penelitian ini tipe BENEFIT (sesuai Tabel 3.1 proposal), jadi cuma
    pakai rumus PDA/NDA versi benefit (Persamaan 2.8 & 2.9)."""
    df = matriks.copy()
    AV = {kol: df[kol].mean() for kol in kolom_kriteria}

    for kol in kolom_kriteria:
        avj = AV[kol]
        df[f'PDA_{kol}'] = np.maximum(0, df[kol] - avj) / avj
        df[f'NDA_{kol}'] = np.maximum(0, avj - df[kol]) / avj

    df['SP'] = sum(bobot[kol] * df[f'PDA_{kol}'] for kol in kolom_kriteria)
    df['SN'] = sum(bobot[kol] * df[f'NDA_{kol}'] for kol in kolom_kriteria)

    max_SP, max_SN = df['SP'].max(), df['SN'].max()
    df['NSP'] = df['SP'] / max_SP if max_SP != 0 else 0.0
    df['NSN'] = 1 - (df['SN'] / max_SN if max_SN != 0 else 0.0)

    df['AS'] = 0.5 * (df['NSP'] + df['NSN'])
    df['rank'] = df['AS'].rank(ascending=False, method='min').astype(int)
    return df.sort_values('AS', ascending=False)


print('Fungsi hitung_edas() siap.')

# Coba langsung ke matriks & bobot contoh
hasil_contoh = hitung_edas(matriks_contoh, ['C1', 'C2', 'C4', 'C5'], bobot_contoh)
print('\nHasil EDAS (contoh, skema 3class, threshold 30), top 10:')
hasil_contoh[['C1', 'C2', 'C4', 'C5', 'SP', 'SN', 'AS', 'rank']].head(10)


Fungsi hitung_edas() siap.

Hasil EDAS (contoh, skema 3class, threshold 30), top 10:


,C1,C2,C4,C5,SP,SN,AS,rank
username,,,,,,,,
tirtacipeng,0.066253,2800000.0,8831.398406,0.028977,2.785953,0.005392,0.747991,1
giapratamamd,0.168188,1300000.0,3467.525180,0.051342,1.327342,0.000000,0.619662,2
ikhsanqthii,0.000677,974300.0,4914.705882,0.077867,1.794678,0.104650,0.600313,3
jamilatussadiyah,0.066116,202500.0,3516.000000,0.062659,1.154421,0.157944,0.511282,4
fauzaannibras,-0.215686,13200.0,545.480447,0.448966,5.546201,0.845891,0.503047,5
itsdoktermuda,0.013100,217300.0,2542.267380,0.095109,1.294651,0.235863,0.478148,6
drkevinmak,0.015198,4200000.0,2525.886266,0.004661,1.387739,0.405087,0.387123,7
doktermedok,0.184089,841200.0,1391.747253,0.010617,0.351775,0.248081,0.385968,8
ekidarehanf,-0.040816,1400000.0,1751.465241,0.010673,0.381839,0.427370,0.283348,9


---
## BLOK F — Jalanin Semua Kombinasi (3 skema x 3 threshold) + Bandingin

**Cell F1**: loop semua kombinasi, simpan hasil tiap kombinasi.
**Cell F2**: simpan semua hasil ke CSV.
**Cell F3**: tampilin ringkasan top-10 tiap kombinasi.
**Cell F4**: bandingin rangking antar kombinasi pake korelasi Spearman (dasar buat sensitivity analysis di proposal, bab 3.1.7).

In [27]:
# ==== Cell F1: Loop semua kombinasi (3 skema x 3 threshold) ====
semua_hasil = {}
ringkasan_kombinasi = []

for nama_skema in SKEMA_C1:
    for threshold in DAFTAR_THRESHOLD_KOMENTAR:
        matriks, sebelum, sesudah = susun_matriks(
            skor_influencer, nama_skema, metrik_akun,
            min_followers=MIN_FOLLOWERS, min_komentar=threshold, min_video=MIN_VIDEO,
        )
        bobot, detail_entropy = hitung_bobot_entropy(matriks, ['C1', 'C2', 'C4', 'C5'])
        hasil = hitung_edas(matriks, ['C1', 'C2', 'C4', 'C5'], bobot)

        key = f'{nama_skema}__min{threshold}'
        semua_hasil[key] = {'matriks': matriks, 'bobot': bobot, 'detail_entropy': detail_entropy, 'hasil': hasil}
        ringkasan_kombinasi.append({
            'skema': nama_skema, 'threshold_komentar': threshold,
            'jumlah_influencer': sesudah, 'top1': hasil.index[0], 'top1_AS': hasil.iloc[0]['AS'],
        })
        print(f'[{key}] {sesudah} influencer -- top1: {hasil.index[0]} (AS={hasil.iloc[0]["AS"]:.3f})')

ringkasan_df = pd.DataFrame(ringkasan_kombinasi)
print('\nRingkasan semua kombinasi:')
ringkasan_df


[3class__min30] 37 influencer -- top1: tirtacipeng (AS=0.748)
[3class__min50] 37 influencer -- top1: tirtacipeng (AS=0.748)
[3class__min100] 33 influencer -- top1: tirtacipeng (AS=1.000)
[3class_hanya_positif__min30] 37 influencer -- top1: fauzaannibras (AS=0.749)
[3class_hanya_positif__min50] 37 influencer -- top1: fauzaannibras (AS=0.749)


C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: invalid value encountered in log
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)
C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: invalid value encountered in log
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)
C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: invalid value encountered in log
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)
C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: divide by zero encountered in log
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)
C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: invalid value encountered in multiply
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)
C:\Users\nesha\AppData\Local\Temp\ipykernel_8364\2617623455.py:12: RuntimeWarning: divide by zero encountered in log
  p_lnp = np.where(p > 0, p * np.log(p), 0.0)
C:\Users\nesha\AppDat

[3class_hanya_positif__min100] 33 influencer -- top1: tirtacipeng (AS=0.997)
[binary__min30] 37 influencer -- top1: fauzaannibras (AS=0.795)
[binary__min50] 37 influencer -- top1: fauzaannibras (AS=0.795)
[binary__min100] 33 influencer -- top1: tirtacipeng (AS=1.000)

Ringkasan semua kombinasi:


,skema,threshold_komentar,jumlah_influencer,top1,top1_AS
0,3class,30,37,tirtacipeng,0.747991
1,3class,50,37,tirtacipeng,0.747991
2,3class,100,33,tirtacipeng,1.000000
3,3class_hanya_positif,30,37,fauzaannibras,0.749398
4,3class_hanya_positif,50,37,fauzaannibras,0.749398
5,3class_hanya_positif,100,33,tirtacipeng,0.996685
6,binary,30,37,fauzaannibras,0.795253
7,binary,50,37,fauzaannibras,0.795253
8,binary,100,33,tirtacipeng,1.000000


In [28]:
# ==== Cell F2: Simpan semua hasil ke CSV ====
for key, data in semua_hasil.items():
    data['hasil'].to_csv(OUT_DIR / f'edas_{key}.csv', encoding='utf-8')

ringkasan_df.to_csv(OUT_DIR / 'ringkasan_semua_kombinasi.csv', index=False, encoding='utf-8')
print(f'Tersimpan {len(semua_hasil)} file edas_<skema>__min<threshold>.csv ke {OUT_DIR}')
print(f'Plus ringkasan_semua_kombinasi.csv')


Tersimpan 9 file edas_<skema>__min<threshold>.csv ke c:\Users\nesha\Tugas_akhir\seleksi-influencer\03_Indobert\Data\hasil_edas
Plus ringkasan_semua_kombinasi.csv


In [29]:
# ==== Cell F3: Tampilin top-10 tiap kombinasi ====
for key, data in semua_hasil.items():
    print(f'\n=== {key} ===')
    print(data['hasil'][['C1', 'C2', 'C4', 'C5', 'AS', 'rank']].head(10).to_string())



=== 3class__min30 ===
                            C1         C2           C4        C5        AS  rank
username                                                                        
tirtacipeng           0.066253  2800000.0  8831.398406  0.028977  0.747991     1
giapratamamd          0.168188  1300000.0  3467.525180  0.051342  0.619662     2
ikhsanqthii           0.000677   974300.0  4914.705882  0.077867  0.600313     3
jamilatussadiyah      0.066116   202500.0  3516.000000  0.062659  0.511282     4
fauzaannibras        -0.215686    13200.0   545.480447  0.448966  0.503047     5
itsdoktermuda         0.013100   217300.0  2542.267380  0.095109  0.478148     6
drkevinmak            0.015198  4200000.0  2525.886266  0.004661  0.387123     7
doktermedok           0.184089   841200.0  1391.747253  0.010617  0.385968     8
ekidarehanf          -0.040816  1400000.0  1751.465241  0.010673  0.283348     9
dokter.harmonospanak  0.326241    19500.0   148.443925  0.031337  0.276921    10

=== 

In [30]:
# ==== Cell F4: Bandingin stabilitas rangking antar kombinasi (Spearman) ====
# Gabungin kolom 'rank' dari semua kombinasi jadi 1 tabel (index=username)
tabel_rank = None
for key, data in semua_hasil.items():
    kolom_rank = data['hasil'][['rank']].rename(columns={'rank': key})
    tabel_rank = kolom_rank if tabel_rank is None else tabel_rank.join(kolom_rank, how='outer')

print('Tabel rank tiap influencer, per kombinasi (NaN = influencer kefilter di kombinasi itu):')
print(tabel_rank.to_string())

print('\nKorelasi rank Spearman antar kombinasi (makin deket 1.0 = makin konsisten/stabil):')
korelasi_rank = tabel_rank.corr(method='spearman')
print(korelasi_rank.round(3).to_string())


Tabel rank tiap influencer, per kombinasi (NaN = influencer kefilter di kombinasi itu):
                       3class__min30  3class__min50  3class__min100  3class_hanya_positif__min30  3class_hanya_positif__min50  3class_hanya_positif__min100  binary__min30  binary__min50  binary__min100
username                                                                                                                                                                                                 
aliphildan                        19             19            19.0                           13                           13                          12.0             14             14            13.0
aymanalts                         20             20            21.0                           12                           12                          11.0             12             12            11.0
bidancynthia                      24             24            27.0                           26        

---
### Cell F5 & F6 -- Diagnosa: kenapa akun yg postingannya dikit bisa rank 1?

Dipicu pertanyaan: kalau diperhatikan, C1 (sentimen) `farhanzubedi` di beberapa skema malah
mendekati 0 / negatif, tapi dia tetep rank 1 di semua kombinasi. **Cell F5** nampilin tabel
bobot entropy ($w_j$) tiap kombinasi -- biar ketahuan kriteria mana yg paling "berbobot" di
tiap skenario (entropy ngasih bobot besar ke kriteria yang datanya paling tersebar/variatif,
jadi kalau C4 atau C5 datanya tersebar jauh karena ada outlier, bobotnya bisa jadi gede dan
outlier itu jadi dominan ke skor akhir). **Cell F6** bedah kontribusi tiap kriteria ke SP/SN
nya 1 username spesifik, buat ngonfirmasi kriteria mana yg paling nge-drag dia naik.

In [31]:
# ==== Cell F5: Tabel bobot entropy (wj) tiap kombinasi ====
tabel_bobot = pd.DataFrame({key: data['bobot'] for key, data in semua_hasil.items()}).T
tabel_bobot.index.name = 'kombinasi'
print('Bobot entropy (wj) per kriteria, tiap kombinasi:')
print(tabel_bobot.round(4).to_string())

print('\nKriteria dengan bobot terbesar per kombinasi:')
for key in tabel_bobot.index:
    kriteria_max = tabel_bobot.loc[key].idxmax()
    print(f'  {key:32s}: {kriteria_max} (wj={tabel_bobot.loc[key, kriteria_max]:.4f})')


Bobot entropy (wj) per kriteria, tiap kombinasi:
                                  C1      C2      C4      C5
kombinasi                                                   
3class__min30                 0.1059  0.2149  0.2908  0.3884
3class__min50                 0.1059  0.2149  0.2908  0.3884
3class__min100                0.2149  0.2261  0.3259  0.2331
3class_hanya_positif__min30   0.0806  0.2210  0.2990  0.3994
3class_hanya_positif__min50   0.0806  0.2210  0.2990  0.3994
3class_hanya_positif__min100  0.1020  0.2586  0.3728  0.2666
binary__min30                 0.0011  0.2401  0.3249  0.4339
binary__min50                 0.0011  0.2401  0.3249  0.4339
binary__min100                0.0010  0.2878  0.4147  0.2966

Kriteria dengan bobot terbesar per kombinasi:
  3class__min30                   : C5 (wj=0.3884)
  3class__min50                   : C5 (wj=0.3884)
  3class__min100                  : C4 (wj=0.3259)
  3class_hanya_positif__min30     : C5 (wj=0.3994)
  3class_hanya_positif__min50

In [32]:
# ==== Cell F6: Bedah kontribusi tiap kriteria ke SP/SN untuk 1 username tertentu ====
def bedah_kontribusi(key, username, kolom_kriteria=['C1', 'C2', 'C4', 'C5']):
    """Nunjukin nilai mentah, PDA, NDA, dan sumbangan ke SP (bobot*PDA) tiap kriteria buat
    1 username di 1 kombinasi -- buat nunjukin kriteria mana yg paling 'ngangkat' dia."""
    data = semua_hasil[key]
    hasil, bobot = data['hasil'], data['bobot']
    if username not in hasil.index:
        print(f'{username} tidak ada di kombinasi {key} (mungkin kefilter).')
        return None
    baris = hasil.loc[username]
    rincian = []
    for kol in kolom_kriteria:
        rincian.append({
            'kriteria': kol,
            'nilai': baris[kol],
            'wj': bobot[kol],
            'PDA': baris[f'PDA_{kol}'],
            'NDA': baris[f'NDA_{kol}'],
            'sumbangan_SP': bobot[kol] * baris[f'PDA_{kol}'],
            'sumbangan_SN': bobot[kol] * baris[f'NDA_{kol}'],
        })
    rincian_df = pd.DataFrame(rincian).set_index('kriteria')
    print(f'[{key}] {username} -- AS={baris["AS"]:.4f}, rank={int(baris["rank"])}')
    print(rincian_df.round(4).to_string())
    print(f'Total SP={baris["SP"]:.4f}  (kriteria penyumbang SP terbesar: '
          f'{rincian_df["sumbangan_SP"].idxmax()})')
    return rincian_df


_ = bedah_kontribusi('3class__min30', 'farhanzubedi')


farhanzubedi tidak ada di kombinasi 3class__min30 (mungkin kefilter).
